In [1]:
import pandas as pd
import numpy as np

In [2]:
# Chapter 10. Data Aggregation and Group Operations

In [3]:
# pandas provides a versatile .groupby() interface, enabling to slice and summarize datasets in a natural way
# query languages impose certain limitations on the kinds of group operations
# with the Python expressiveness can express as custom Python functions 

In [5]:
# 10.1 How to think About Group Operations

In [6]:
# group operations : split-apply-combine
# Series, DataFrame is split into groups based on one or more keys that you provide
# splitting is performed on a particular axis of an object
# once split, function is applied to each group, producing a new value
# results of all those function applications are combined into a result object

In [8]:
# each grouping key can take many forms, and the keys do not have to be all the same type:
# - A list or array of values that is the same length as the axis being grouped
# - A value indicating a column name in a DataFrame
# - A dictionary or Series giving a correspondence between the values on the axis being grouped and the group names
# - A function to be invoked on the axis index or the individuals labels in the index

In [10]:
df = pd.DataFrame({"key1":["a","a",None,"b","b","a",None],
                  "key2":pd.Series([1,2,1,2,1,None,1],dtype="Int64"),
                  "data1":np.random.standard_normal(7),
                  "data2":np.random.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,0.666066,1.546214
1,a,2,1.253839,0.521858
2,None,1,-0.846390,-0.613611
3,b,2,-0.302440,-1.316962
4,b,1,-1.343540,1.208730
5,a,<NA>,1.419217,-0.197108
6,None,1,-0.423734,-0.484917


In [11]:
# suppose you want to compute the mean of the data1 column using the labels from key1

In [12]:
# method 1 : access data1 and call groupby with the column (a Series) at key1:

grouped = df["data1"].groupby(df["key1"])
grouped

In [13]:
# grouped is now a special "GroupBy" object
# to compute group means we can call the GroupBy's .mean() method:
grouped.mean()

# here the data (a Series) has been aggregated by splitting the data on the group key;
# producing a new Series that is now indexed by the unique values in the key1 column
# the result index has the name "key1" because the DataFrame column df["key1"] did

key1
a    1.113041
b   -0.822990
Name: data1, dtype: float64

In [14]:
# if instead passed multiple arrays as a list, we'd get something different:
means = df["data1"].groupby([df["key1"],df["key2"]]).mean()
means

key1  key2
a     1       0.666066
      2       1.253839
b     1      -1.343540
      2      -0.302440
Name: data1, dtype: float64

In [16]:
# here we grouped the data using two keys, and the resulting Series now has a heirarchical index;
# which consist of the unique pairs of keys observed:
means.unstack()

key2,1,2
key1,,
a,0.666066,1.253839
b,-1.343540,-0.302440


In [20]:
# in this example, the group keys are all Series, though they could be any arrays of the right lenght:
states = np.array(["OH","CA","CA","OH","OH","CA","OH"])
years = [2005,2005,2006,2005,2006,2005,2006]
df["data1"].groupby([states,years]).mean()

CA  2005    1.336528
    2006   -0.846390
OH  2005    0.181813
    2006   -0.883637
Name: data1, dtype: float64

In [22]:
# frequently, the grouping information is the same DataFrame as the data you want to work on
# in this caes, you can pass column names as the group keys:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,1.113041,0.623655
b,1.5,-0.822990,-0.054116


In [23]:
df.groupby("key2").mean()

# there is no key1 column in the result since df["key1"] is not numeric data
# non-numeric data is said to be a nuisance column which is automatically excluded from result
# by default, all of the numeric columns are aggregated

,data1,data2
key2,,
1,-0.486899,0.414104
2,0.475699,-0.397552


In [24]:
df.groupby(["key1","key2"]).mean()

data1     data2
key1 key2                    
a    1     0.666066  1.546214
     2     1.253839  0.521858
b    1    -1.343540  1.208730
     2    -0.302440 -1.316962

In [25]:
# regardless of the objective in using .groupby(), generally useful GroupBy method is .size()
# .size() returns a Series containing group sizes:

df.groupby(["key1","key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [27]:
# any missing values in a group key are excluded from the result by default
# this behavior can be disabled by passing dropna=False to .groupby:

df.groupby("key1",dropna=False).size()

key1
a      3
b      2
NaN    2
dtype: int64

In [28]:
df.groupby(["key1","key2"],dropna=False).size()

key1  key2
a     1       1
      2       1
      NaN     1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [29]:
# a group function similar to .size() is .count(), which computes the number of non-null values in each group:

df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2
